In [14]:
import torch
from torch.utils.data import DataLoader, TensorDataset

class LinearRegressionScratch:
    """Tensor와 자동미분만 사용한 선형회귀 모델."""

    def __init__(
        self,
        num_inputs,
        learning_rate,
        sigma=0.01,
    ):
        self.num_inputs = num_inputs
        self.learning_rate = learning_rate

        # w.shape = (num_inputs, 1)
        self.w = torch.normal(
            mean=0.0,
            std=sigma,
            size=(num_inputs, 1),
            requires_grad=True,
        )

        # b.shape = (1,)
        self.b = torch.zeros(
            1,
            requires_grad=True,
        )

    def forward(self, X):
        # (B, d) @ (d, 1) + (1) -> (B, 1)
        return X @ self.w + self.b

    def __call__(self, X):
        return self.forward(X)


def squared_loss(predictions, labels):
    """미니배치의 평균 제곱손실을 계산한다."""

    labels = labels.reshape(predictions.shape)
    errors = predictions - labels
    per_example_loss = 0.5 * errors.pow(2)

    return per_example_loss.mean()

In [15]:
# Cell 3
class SGD:
    """미니배치 확률적 경사하강법."""

    def __init__(self, parameters, learning_rate):
        # 갱신할 weight와 bias를 저장한다.
        self.parameters = list(parameters)
        self.learning_rate = learning_rate

    def zero_grad(self):
        # PyTorch는 gradient를 누적하므로
        # 새로운 backward 전에 기존 gradient를 제거한다.
        for parameter in self.parameters:
            if parameter.grad is not None:
                parameter.grad.zero_()

    def step(self):
        # Parameter 갱신 과정은 계산 그래프에 기록하지 않는다.
        with torch.no_grad():
            for parameter in self.parameters:
                if parameter.grad is not None:
                    parameter -= (
                        self.learning_rate
                        * parameter.grad
                    )

In [16]:
# 정답을 알고 있는 합성 회귀 데이터를 생성한다.
true_weights = torch.tensor([2.0, -3.4])
true_bias = 4.2

number_of_examples = 1000
number_of_features = 2
batch_size = 32

# X.shape = (1000, 2)
training_features = torch.randn(
    number_of_examples,
    number_of_features,
)

# epsilon ~ N(0, 0.01^2)
noise = (
    torch.randn(number_of_examples, 1)
    * 0.01
)

# y = Xw + b + epsilon
# y.shape = (1000, 1)
training_labels = (
    training_features @ true_weights.reshape(-1, 1) 
    + true_bias 
    + noise
)

training_dataset = TensorDataset(
    training_features,
    training_labels,
)

training_loader = DataLoader(
    dataset=training_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

print("Features shape:", training_features.shape)
print("Labels shape:", training_labels.shape)
print("Number of batches:", len(training_loader))

Features shape: torch.Size([1000, 2])
Labels shape: torch.Size([1000, 1])
Number of batches: 32


In [19]:
# Cell 5
model = LinearRegressionScratch(
    num_inputs=2,
    learning_rate=0.03,
)

optimizer = SGD(
    parameters=[model.w, model.b],
    learning_rate=model.learning_rate,
)

number_of_epochs = 10
loss_history = []

for epoch in range(number_of_epochs):
    total_loss = 0.0
    total_samples = 0

    for batch_features, batch_labels in training_loader:
        # 1. 이전 batch의 gradient를 제거한다.
        optimizer.zero_grad()

        # 2. 현재 parameter로 예측한다.
        predictions = model(batch_features)

        # 3. 현재 batch의 평균 loss를 계산한다.
        loss = squared_loss(
            predictions,
            batch_labels,
        )

        # 4. w.grad와 b.grad를 계산한다.
        loss.backward()

        # 5. 계산된 gradient로 w와 b를 갱신한다.
        optimizer.step()

        current_batch_size = batch_labels.shape[0]

        total_loss += loss.item() * current_batch_size
        total_samples += current_batch_size

    epoch_loss = total_loss / total_samples
    loss_history.append(epoch_loss)

    print(
        f"Epoch {epoch + 1} "
        f"| Mean loss: {epoch_loss:.6f}"
    )

Epoch 1 | Mean loss: 7.507622
Epoch 2 | Mean loss: 1.193683
Epoch 3 | Mean loss: 0.196071
Epoch 4 | Mean loss: 0.031069
Epoch 5 | Mean loss: 0.005137
Epoch 6 | Mean loss: 0.000858
Epoch 7 | Mean loss: 0.000180
Epoch 8 | Mean loss: 0.000073
Epoch 9 | Mean loss: 0.000055
Epoch 10 | Mean loss: 0.000052


In [21]:
# Cell 6
learned_weights = model.w.detach().reshape(
    true_weights.shape
)

learned_bias = model.b.detach()

weight_error = true_weights - learned_weights
bias_error = true_bias - learned_bias

print("True weights:", true_weights)
print("Learned weights:", learned_weights)
print("Weight error:", weight_error)

print("\nTrue bias:", true_bias)
print("Learned bias:", learned_bias)
print("Bias error:", bias_error)

print("\nLoss history:", loss_history)

True weights: tensor([ 2.0000, -3.4000])
Learned weights: tensor([ 1.9995, -3.3995])
Weight error: tensor([ 0.0005, -0.0005])

True bias: 4.2
Learned bias: tensor([4.1998])
Bias error: tensor([0.0002])

Loss history: [7.507622243881226, 1.1936827499866485, 0.19607115364074706, 0.03106949020922184, 0.005136579796671867, 0.0008576491386629641, 0.0001801988240913488, 7.251197716686874e-05, 5.535090516787023e-05, 5.2318271656986324e-05]
